# Newspaper Article Retrieval — 2024 US Presidential Election

Retrieves election-related news articles from politically diverse outlets for **4 months before** the election: **Jul 5 – Nov 4, 2024**.

| Source | Leaning | Method |
|---|---|---|
| The Guardian | Democratic | Guardian Open Platform API (free) |
| New York Times | Democratic | NYT Article Search API (free) |
| NBC News | Democratic | GDELT DOC 2.0 API (no key needed) |
| Fox News | Republican | GDELT DOC 2.0 API (no key needed) |
| Breitbart | Republican | GDELT DOC 2.0 API (no key needed) |
| NY Post | Republican | GDELT DOC 2.0 API (no key needed) |
| Daily Caller | Republican | GDELT DOC 2.0 API (no key needed) |

**API keys needed:**
- Guardian: https://open-platform.theguardian.com/access/
- NYT: https://developer.nytimes.com/get-started
- GDELT: no key required

In [8]:
import requests
import pandas as pd
import time
import os

# ── API Keys ─────────────────────────────────────────────────────────────────
GUARDIAN_API_KEY = "47346645-10a7-4288-aa66-7a7191f8f9eb"
NYT_API_KEY      = "QvQ4PUkpkCtnwmwkO1bLJNgfbkmhCK2NRh5PD4ypHZeAGpNr"

# ── Date range: 3 months before the election (Nov 5, 2024) ───────────────────
START_DATE = "2024-07-05"
END_DATE   = "2024-11-04"

# ── Search query ──────────────────────────────────────────────────────────────
QUERY = "Trump OR Harris presidential election 2024"

# ── Final columns ─────────────────────────────────────────────────────────────
COLS = ["source", "leaning", "date", "title", "url"]

os.makedirs("data", exist_ok=True)
print(f"Date range: {START_DATE} to {END_DATE}")

Date range: 2024-07-05 to 2024-11-04


## 1. The Guardian API (Democratic-leaning)

In [9]:
def fetch_guardian(query, from_date, to_date, api_key, max_pages=15):
    articles = []
    for page in range(1, max_pages + 1):
        params = {
            "q":           query,
            "from-date":   from_date,
            "to-date":     to_date,
            "api-key":     api_key,
            "page-size":   200,
            "page":        page,
            "show-fields": "headline",
            "order-by":    "oldest",
        }
        r = requests.get("https://content.guardianapis.com/search", params=params, timeout=30)
        r.raise_for_status()
        response = r.json()["response"]
        results  = response.get("results", [])
        if not results:
            break
        for item in results:
            articles.append({
                "source":  "The Guardian",
                "leaning": "Democratic",
                "date":    item["webPublicationDate"][:10],
                "title":   item.get("fields", {}).get("headline", item.get("webTitle", "")),
                "url":     item.get("webUrl", ""),
            })
        total_pages = response.get("pages", 1)
        print(f"  page {page}/{min(max_pages, total_pages)} — {len(results)} articles")
        if page >= total_pages:
            break
        time.sleep(0.5)
    return pd.DataFrame(articles)

print("Fetching Guardian articles...")
df_guardian = fetch_guardian(QUERY, START_DATE, END_DATE, GUARDIAN_API_KEY)
print(f"Total: {len(df_guardian)} articles")
df_guardian.head(3)

Fetching Guardian articles...
  page 1/15 — 200 articles
  page 2/15 — 200 articles
  page 3/15 — 200 articles
  page 4/15 — 200 articles
  page 5/15 — 200 articles
  page 6/15 — 200 articles
  page 7/15 — 200 articles
  page 8/15 — 200 articles
  page 9/15 — 200 articles
  page 10/15 — 200 articles
  page 11/15 — 200 articles
  page 12/15 — 200 articles
  page 13/15 — 200 articles
  page 14/15 — 200 articles
  page 15/15 — 200 articles
Total: 3000 articles


,source,leaning,date,title,url
0,The Guardian,Democratic,2024-07-05,Ukraine war briefing: Ukrainian army confirms ...,https://www.theguardian.com/world/article/2024...
1,The Guardian,Democratic,2024-07-05,Nigel Farage hails Reform UK’s ‘almost unbelie...,https://www.theguardian.com/politics/article/2...
2,The Guardian,Democratic,2024-07-05,"The seconds ticked by to 10pm, when the people...",https://www.theguardian.com/politics/article/2...


## 2. New York Times Article Search API (Democratic-leaning)

In [10]:
def fetch_nyt(query, begin_date, end_date, api_key, max_pages=10):
    """10 articles per page → max_pages=10 gives ~100 articles."""
    base_url  = "https://api.nytimes.com/svc/search/v2/articlesearch.json"
    articles  = []
    begin_fmt = begin_date.replace("-", "")
    end_fmt   = end_date.replace("-", "")

    for page in range(max_pages):
        params = {
            "q":          query,
            "begin_date": begin_fmt,
            "end_date":   end_fmt,
            "api-key":    api_key,
            "page":       page,
            "sort":       "oldest",
            "fl":         "headline,pub_date,web_url",
        }
        for attempt in range(3):
            r = requests.get(base_url, params=params, timeout=30)
            if r.status_code in (401, 429):
                wait = 15 * (attempt + 1)
                print(f"  Rate limited (attempt {attempt+1}), waiting {wait}s...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            break
        else:
            print(f"  Skipping page {page+1} after 3 failed attempts.")
            continue

        docs = r.json()["response"]["docs"]
        if not docs:
            break

        for doc in docs:
            articles.append({
                "source":  "New York Times",
                "leaning": "Democratic",
                "date":    doc.get("pub_date", "")[:10],
                "title":   doc.get("headline", {}).get("main", ""),
                "url":     doc.get("web_url", ""),
            })

        print(f"  page {page + 1} — {len(docs)} articles")
        time.sleep(12)  # 12s gap → safely under 10 req/min

    return pd.DataFrame(articles)

print("Fetching NYT articles (~100 articles)...")
df_nyt = fetch_nyt(QUERY, START_DATE, END_DATE, NYT_API_KEY, max_pages=10)
print(f"Total: {len(df_nyt)} articles")
df_nyt.head(3)

Fetching NYT articles (~100 articles)...
  page 1 — 10 articles
  page 2 — 10 articles
  page 3 — 10 articles
  page 4 — 10 articles
  page 5 — 10 articles
  page 6 — 10 articles
  page 7 — 10 articles
  page 8 — 10 articles
  page 9 — 10 articles
  page 10 — 10 articles
Total: 100 articles


,source,leaning,date,title,url
0,New York Times,Democratic,2024-07-05,Is Kamala Harris Underrated?,https://www.nytimes.com/2024/07/05/opinion/ezr...
1,New York Times,Democratic,2024-07-05,These Voters Supported Biden in 2020. Now They...,https://www.nytimes.com/2024/07/05/us/election...
2,New York Times,Democratic,2024-07-05,"Biden Campaign, Sticking to Its Playbook, Will...",https://www.nytimes.com/2024/07/05/us/politics...


## 3. GDELT DOC 2.0 — Fox News, Breitbart, NY Post (Republican) + NBC News (Democratic)

GDELT monitors global news 24/7 across thousands of outlets. No API key required.

In [12]:
def fetch_gdelt_domain(keyword, domain, start_dt, end_dt, max_records=250):
    params = {
        "query":         f"{keyword} domain:{domain}",
        "mode":          "artlist",
        "maxrecords":    max_records,
        "startdatetime": start_dt,
        "enddatetime":   end_dt,
        "format":        "json",
        "sort":          "DateAsc",
    }
    for attempt in range(5):
        try:
            r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                             params=params, timeout=60)
            if r.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"    Rate limited, waiting {wait}s (attempt {attempt+1}/5)...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            if not r.text.strip():
                return []
            return r.json().get("articles", [])
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f"    Error: {e} — retrying in {wait}s...")
            time.sleep(wait)
    print(f"    Failed after 5 attempts for {domain}, skipping.")
    return []


SOURCES = {
    # Democratic-leaning
    "nbcnews.com":     ("NBC News",     "Democratic"),
    # Republican-leaning
    "foxnews.com":     ("Fox News",     "Republican"),
    "breitbart.com":   ("Breitbart",    "Republican"),
    "nypost.com":      ("NY Post",      "Republican"),
    "dailycaller.com": ("Daily Caller", "Republican"),
}

START_GDELT = START_DATE.replace("-", "") + "000000"
END_GDELT   = END_DATE.replace("-", "")   + "235959"
keyword     = "Trump Harris election"

gdelt_rows = []
print("Fetching GDELT articles...")
for domain, (source_name, leaning) in SOURCES.items():
    results = fetch_gdelt_domain(keyword, domain, START_GDELT, END_GDELT)
    print(f"  [{source_name}]: {len(results)} articles")
    for item in results:
        raw_date = item.get("seendate", "")[:8]
        try:
            date_fmt = pd.to_datetime(raw_date, format="%Y%m%d").strftime("%Y-%m-%d")
        except Exception:
            date_fmt = ""
        gdelt_rows.append({
            "source":  source_name,
            "leaning": leaning,
            "date":    date_fmt,
            "title":   item.get("title", ""),
            "url":     item.get("url", ""),
        })
    time.sleep(10)  # wait between sources to avoid rate limiting

df_gdelt = pd.DataFrame(gdelt_rows)
print(f"\nGDELT total: {len(df_gdelt)} articles")
df_gdelt.head(3)

Fetching GDELT articles...
    Rate limited, waiting 30s (attempt 1/5)...
  [NBC News]: 250 articles
  [Fox News]: 250 articles
  [Breitbart]: 250 articles
  [NY Post]: 250 articles
  [Daily Caller]: 250 articles

GDELT total: 1250 articles


,source,leaning,date,title,url
0,NBC News,Democratic,2024-08-07,Chuck Todd : The campaign reset is complete,https://www.nbcnews.com/politics/elections/chu...
1,NBC News,Democratic,2024-08-07,Republicans start tearing down Walz by going a...,https://www.nbcnews.com/politics/2024-election...
2,NBC News,Democratic,2024-08-08,Kamala Harris rally in Michigan interrupted by...,https://www.nbcnews.com/politics/2024-election...


## 4. Combine All Sources & Save

In [16]:
df_all = pd.concat([df_guardian, df_nyt, df_gdelt], ignore_index=True)[COLS]

# Normalise dates and filter range
df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
mask = (df_all["date"] >= START_DATE) & (df_all["date"] <= END_DATE)
df_all = df_all[mask].drop_duplicates(subset="url").sort_values("date").reset_index(drop=True)

print(f"Before keyword filter : {len(df_all)} articles")

# ── Keyword filter: keep only US election-relevant articles ──────────────────
US_KEYWORDS = [
    "trump", "harris", "kamala", "donald",
    "biden", "presidential", "white house",
    "republican", "democrat", "gop",
    "swing state", "battleground", "electoral college",
    "maga", "election day", "november 5",
    "poll", "debate", "campaign", "ballot",
    "oval office", "nominee", "vice president",
]

pattern = "|".join(US_KEYWORDS)
mask_kw = df_all["title"].str.lower().str.contains(pattern, na=False)
df_all = df_all[mask_kw].reset_index(drop=True)

print(f"After keyword filter  : {len(df_all)} articles")
print(f"Removed               : {(~mask_kw).sum()} irrelevant articles\n")
print("=== Combined dataset ===")
print(df_all.groupby(["source", "leaning"]).size().to_string())
print(f"\nDate range: {df_all['date'].min().date()} → {df_all['date'].max().date()}")
df_all.head()

Before keyword filter : 4350 articles
After keyword filter  : 1752 articles
Removed               : 2598 irrelevant articles

=== Combined dataset ===
source          leaning   
Breitbart       Republican    208
Daily Caller    Republican    212
Fox News        Republican    208
NBC News        Democratic    209
NY Post         Republican    211
New York Times  Democratic     85
The Guardian    Democratic    619

Date range: 2024-07-05 → 2024-10-23


,source,leaning,date,title,url
0,The Guardian,Democratic,2024-07-05,How Labour did it: inside the campaign that le...,https://www.theguardian.com/politics/article/2...
1,The Guardian,Democratic,2024-07-05,Joe Biden set for media blitz as more lawmaker...,https://www.theguardian.com/us-news/article/20...
2,The Guardian,Democratic,2024-07-05,Warning signs: a history of Joe Biden’s verbal...,https://www.theguardian.com/us-news/article/20...
3,The Guardian,Democratic,2024-07-05,Intra-Maga row for Missouri’s attorney general...,https://www.theguardian.com/us-news/article/20...
4,The Guardian,Democratic,2024-07-05,Donald Trump claims to ‘know nothing’ about Pr...,https://www.theguardian.com/us-news/article/20...


In [17]:
df_all.to_csv("data/newspaper_articles.csv", index=False)
df_all[df_all["leaning"] == "Democratic"].to_csv("data/newspaper_democratic.csv", index=False)
df_all[df_all["leaning"] == "Republican"].to_csv("data/newspaper_republican.csv", index=False)

print("Saved:")
print("  data/newspaper_articles.csv   (all sources)")
print("  data/newspaper_democratic.csv (Guardian, NYT, NBC News)")
print("  data/newspaper_republican.csv (Fox News, Breitbart, NY Post, Daily Caller)")

Saved:
  data/newspaper_articles.csv   (all sources)
  data/newspaper_democratic.csv (Guardian, NYT, NBC News)
  data/newspaper_republican.csv (Fox News, Breitbart, NY Post, Daily Caller)
